# Validation: NB versus ZINB Likelihood

- **Purpose:** Retrain both winners with `gene_likelihood="nb"` to test sensitivity to the count likelihood.
- **Inputs:** Processed H5ADs; `models/hpo/best_models_primary_100.csv`; winner configs and models.
- **Outputs:** Metrics printed in the notebook; artifacts under `models/validation/nb_sensitivity/`.
- **Result:** Joint scores are essentially unchanged (0.681 versus 0.679); scArches drops (0.489 versus 0.520), so ZINB is kept.


Retrains both HPO winners with only `gene_likelihood` switched from `zinb`
to `nb`. Everything else is identical to the seed-0 HPO runs:
same architecture, stratified split, KL warmup, early stopping, evaluation.

- joint winner: config 10 / hvg3000; scArches winner: config 47 / hvg10000.
- The comparison runs on scIB + kNN metrics + per-line line scores. ELBO is
  NOT compared: the two likelihood families have different scales.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import anndata as ad
import scvi

from src import hpo
from src import hpo_pp_utils as ppu
from src import model_io
from src.utils import load_config

CFG = load_config()
PATHS = CFG["paths"]
HPO_CFG = CFG["hpo"]

adata_tumor = ad.read_h5ad(ROOT / PATHS["processed"]["pp_tumor"])
adata_cl = ad.read_h5ad(ROOT / PATHS["processed"]["pp_cell_line"])
print(f"tumor {adata_tumor.shape}, cell line {adata_cl.shape}")

tumor (108131, 28617), cell line (2162, 28617)


# HPO winners (fixed k=100)

In [2]:
best = pd.read_csv(ROOT / "models" / "hpo" / "best_models_primary_100.csv")
best[["config_id", "num_hvg", "approach", "run_dir", "k100_mean_line_score"]]

,config_id,num_hvg,approach,run_dir,k100_mean_line_score
0,10,3000,joint,models/hpo/runs/hvg3000/config_10/joint,0.679064
1,47,10000,scarches,models/hpo/runs/hvg10000/config_47/scarches,0.519707


In [3]:
JOINT_RUN = ROOT / best.loc[best["approach"] == "joint", "run_dir"].iloc[0]
SCARCHES_RUN = ROOT / best.loc[best["approach"] == "scarches", "run_dir"].iloc[0]
joint_cfg = json.loads((JOINT_RUN / "config.json").read_text())
scarches_cfg = json.loads((SCARCHES_RUN / "config.json").read_text())
print(JOINT_RUN)
print(SCARCHES_RUN)

/mnt/c/Users/marku/github/scMapping-public/models/hpo/runs/hvg3000/config_10/joint
/mnt/c/Users/marku/github/scMapping-public/models/hpo/runs/hvg10000/config_47/scarches


# Joint winner with gene_likelihood = nb

In [4]:
torch.set_float32_matmul_precision("high")
nb_joint_cfg = {**joint_cfg, "gene_likelihood": "nb"}
hpo.set_seed(nb_joint_cfg["seed"])

combined_3k = model_io.build_combined_adata_for_hvg(
    adata_tumor, adata_cl, nb_joint_cfg["num_hvg"], hvg_dir=JOINT_RUN.parents[1]
)
nb_joint_dir = ROOT / "models" / "validation" / "nb_sensitivity" / "joint_hvg3000"
if (nb_joint_dir / "joint_scanvi").exists():
    print(f"[load] {nb_joint_dir / 'joint_scanvi'}")
    nb_joint = scvi.model.SCANVI.load(str(nb_joint_dir / "joint_scanvi"), adata=combined_3k)
else:
    nb_joint = hpo.train_joint_scanvi(combined_3k, nb_joint_cfg, nb_joint_dir)
if hpo.is_run_done(nb_joint_dir):
    print("[skip] evaluation already done")
else:
    hpo.evaluate(
        nb_joint, combined_3k, nb_joint_cfg, nb_joint_dir, "joint",
        scib_n_cells=HPO_CFG["scib_n_cells"],
    )
    hpo.mark_run_done(nb_joint_dir)

Seed set to 0


[load] /mnt/c/Users/marku/github/scMapping-public/models/validation/nb_sensitivity/joint_hvg3000/joint_scanvi
INFO     File                                                                                                      
         /mnt/c/Users/marku/github/scMapping-public/models/validation/nb_sensitivity/joint_hvg3000/joint_scanvi/mod
         el.pt already downloaded                                                                                  
[skip] evaluation already done


# scArches winner with gene_likelihood = nb

In [5]:
hpo.set_seed(scarches_cfg["seed"])
hvg_dir_10k = SCARCHES_RUN.parents[1]
hvg_genes = (hvg_dir_10k / "hvg_genes.txt").read_text().splitlines()
tumor_hvg = adata_tumor[:, hvg_genes].copy()
cl_hvg = adata_cl[:, hvg_genes].copy()
tumor_hvg.layers["counts"] = tumor_hvg.X.copy()
cl_hvg.layers["counts"] = cl_hvg.X.copy()
combined_10k = ppu.build_combined_adata(tumor_hvg, cl_hvg)
print(f"combined {combined_10k.shape}")

Seed set to 0


combined (110293, 10000)


In [6]:
nb_scarches_cfg = {**scarches_cfg, "gene_likelihood": "nb"}
hpo.set_seed(nb_scarches_cfg["seed"])

nb_scarches_dir = ROOT / "models" / "validation" / "nb_sensitivity" / "scarches_hvg10000"
if (nb_scarches_dir / "query").exists():
    print(f"[load] {nb_scarches_dir / 'query'}")
    nb_query = scvi.model.SCANVI.load(str(nb_scarches_dir / "query"), adata=combined_10k)
else:
    _, nb_query = hpo.train_scarches(tumor_hvg, cl_hvg, nb_scarches_cfg, nb_scarches_dir)
if hpo.is_run_done(nb_scarches_dir):
    print("[skip] evaluation already done")
else:
    hpo.evaluate(
        nb_query, combined_10k, nb_scarches_cfg, nb_scarches_dir, "scarches",
        scib_n_cells=HPO_CFG["scib_n_cells"],
    )
    hpo.mark_run_done(nb_scarches_dir)

Seed set to 0


[load] /mnt/c/Users/marku/github/scMapping-public/models/validation/nb_sensitivity/scarches_hvg10000/query
INFO     File                                                                                                      
         /mnt/c/Users/marku/github/scMapping-public/models/validation/nb_sensitivity/scarches_hvg10000/query/model.
         pt already downloaded                                                                                     
[skip] evaluation already done


# Comparison
`n_epochs`/`best_epoch` are training diagnostics (comparable within each pair).
Score columns carry the NB vs ZINB decision.

In [7]:
from src.validation import line_table, summary_row


In [8]:
for run in (JOINT_RUN, nb_joint_dir, SCARCHES_RUN, nb_scarches_dir):
    s = json.loads((run / "eval_summary.json").read_text())
    print(f"{run}: n_epochs={s['n_epochs']} best_epoch={s['best_epoch']}")

comparison = pd.DataFrame({
    "joint ZINB (HPO)": summary_row(JOINT_RUN),
    "joint NB": summary_row(nb_joint_dir),
    "scArches ZINB (HPO)": summary_row(SCARCHES_RUN),
    "scArches NB": summary_row(nb_scarches_dir),
}).T
comparison.round(4)

/mnt/c/Users/marku/github/scMapping-public/models/hpo/runs/hvg3000/config_10/joint: n_epochs=75 best_epoch=59
/mnt/c/Users/marku/github/scMapping-public/models/validation/nb_sensitivity/joint_hvg3000: n_epochs=91 best_epoch=75
/mnt/c/Users/marku/github/scMapping-public/models/hpo/runs/hvg10000/config_47/scarches: n_epochs=400 best_epoch=399
/mnt/c/Users/marku/github/scMapping-public/models/validation/nb_sensitivity/scarches_hvg10000: n_epochs=336 best_epoch=335


,mean_line_score,mean_mixing,mean_specificity,mean_coverage,mean_line_purity,scib_Total,scib_Batch_correction,scib_Bio_conservation,n_epochs
joint ZINB (HPO),0.6791,0.6923,0.9496,1.0000,0.5224,0.5929,0.5999,0.5883,75.0
joint NB,0.6808,0.6723,0.9558,1.0000,0.5298,0.6098,0.5976,0.6180,91.0
scArches ZINB (HPO),0.5197,0.3729,0.9468,0.9890,0.4442,0.5657,0.5649,0.5662,400.0
scArches NB,0.4885,0.3217,0.8988,0.9399,0.4567,0.5470,0.5520,0.5437,336.0


In [9]:
pd.DataFrame({
    "joint ZINB": line_table(JOINT_RUN)["line_score"],
    "joint NB": line_table(nb_joint_dir)["line_score"],
    "scArches ZINB": line_table(SCARCHES_RUN)["line_score"],
    "scArches NB": line_table(nb_scarches_dir)["line_score"],
}).round(4)

,joint ZINB,joint NB,scArches ZINB,scArches NB
cell_line,,,,
NCIH747,0.5159,0.5427,0.4360,0.4037
LS1034,0.7719,0.7793,0.6312,0.5703
RCM1,0.6536,0.6301,0.4706,0.4902
CCK81,0.6314,0.6452,0.7066,0.7021
SNUC4,0.7122,0.7219,0.5370,0.4883
CL34,0.6493,0.6477,0.5354,0.4725
LS180,0.7547,0.7274,0.4779,0.4364
OUMS23,0.6438,0.7267,0.3901,0.3910
HCC56,0.7049,0.6769,0.4506,0.4510
